# Session 4 — Remote sensing foundations and spectral indices
*Séance 4 — Fondements de la télédétection et indices spectraux*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/04_teledetection_indices.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Describe the components of an EO system and the difference between passive and active sensors.
2. Compare the spatial, spectral, temporal and radiometric resolutions of Sentinel-2, Landsat 8/9 and MODIS.
3. Read spectral signatures and build true- and false-colour composites.
4. Compute and interpret NDVI, NDWI and NDBI with NumPy.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

### What is remote sensing?
Remote sensing is the acquisition of information about the Earth's surface **without physical contact**, by measuring
electromagnetic radiation reflected or emitted by the surface. An EO system has five components: an **energy source**
(the Sun, or the sensor itself), the **atmosphere** the radiation crosses twice, the **target**, the **sensor** on a satellite
or drone, and the **ground segment** that receives, corrects and distributes the data.

| Sensor type | Energy source | Examples | Strengths |
|---|---|---|---|
| Passive optical | Sunlight reflected by the surface | Sentinel-2 MSI, Landsat 8/9 OLI, MODIS | Spectral detail (vegetation, water, minerals) |
| Passive thermal | Heat emitted by the surface | Landsat TIRS, ECOSTRESS | Surface temperature, evapotranspiration |
| Active radar (SAR) | Microwave pulse sent by the sensor | Sentinel-1, RADARSAT Constellation | Works day and night and through clouds (floods, deformation) |
| Active lidar | Laser pulse | GEDI, ICESat-2, airborne lidar | Vegetation height, terrain |

### Resolutions of the sensors used in this course

| | Sentinel-2 (MSI) | Landsat 8/9 (OLI) | MODIS (Terra/Aqua) |
|---|---|---|---|
| Spatial | 10, 20, 60 m | 30 m (15 m panchromatic) | 250, 500, 1000 m |
| Spectral | 13 bands, 443–2190 nm | 9 bands + 2 thermal | 36 bands |
| Temporal (revisit) | 5 days (2 satellites) | 8 days (2 satellites) | 1–2 days |
| Radiometric | 12 bit | 12 bit (14 bit Landsat 9) | 12 bit |
| Free archive since | 2015 | 1972 (Landsat programme) | 2000 |

**Processing levels.** *Level-1C* = top-of-atmosphere (TOA) reflectance, geometrically corrected; *Level-2A* = **surface
reflectance**, corrected for the atmosphere — the product to use for multi-date analysis and machine learning.

### Spectral signatures
Each surface reflects light differently at each wavelength. Let's plot typical values.

In [ ]:
import numpy as np

BANDS = ["B2", "B3", "B4", "B8", "B11"]                       # blue, green, red, NIR, SWIR1
WAVELENGTH_NM = [490, 560, 665, 842, 1610]
# typical surface reflectance (0-1) of four land-cover types for these bands
LIBRARY = {"water":      [0.06, 0.07, 0.05, 0.03, 0.01],
           "vegetation": [0.03, 0.07, 0.04, 0.40, 0.20],
           "bare soil":  [0.10, 0.14, 0.18, 0.25, 0.32],
           "built-up":   [0.12, 0.13, 0.14, 0.18, 0.30]}


def synthetic_scene(size=200, seed=0):
    """Return (cube, labels): a Sentinel-2-like reflectance cube (bands, rows, cols) and the true class of each pixel."""
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:size, 0:size]
    labels = np.full((size, size), 2)                                      # bare soil background
    labels[(x - 0.3 * size) ** 2 + (y - 0.35 * size) ** 2 < (0.15 * size) ** 2] = 0   # a lake
    labels[(y > 0.55 * size) & (x < 0.6 * size)] = 1                                  # fields
    labels[(x > 0.68 * size) & (y > 0.2 * size) & (y < 0.8 * size)] = 3               # a town
    names = list(LIBRARY)
    mean = np.array([LIBRARY[n] for n in names])[labels]                   # rows, cols, bands
    cube = mean * rng.lognormal(0, 0.08, mean.shape)
    return np.moveaxis(cube, -1, 0).astype("float32"), labels

import matplotlib.pyplot as plt
for name, refl in LIBRARY.items():
    plt.plot(WAVELENGTH_NM, refl, marker="o", label=name)
plt.xlabel("Wavelength (nm)"); plt.ylabel("Surface reflectance"); plt.title("Typical spectral signatures")
plt.legend(); plt.grid(alpha=.3); plt.show()

Vegetation reflects strongly in the **near-infrared (NIR, B8)** and absorbs **red (B4)** for photosynthesis; water absorbs
almost everything beyond green; built-up areas and bare soil are bright in the **short-wave infrared (SWIR, B11)**.

### Composites
We simulate a 200 × 200 pixel Sentinel-2 scene (10 m pixels, i.e. 2 km × 2 km) with a lake, fields, bare soil and a town.

In [ ]:
cube, truth = synthetic_scene()
b = {name: cube[i] for i, name in enumerate(BANDS)}


def stretch(img):
    lo, hi = np.percentile(img, (2, 98))
    return np.clip((img - lo) / (hi - lo), 0, 1)


true_colour = np.dstack([stretch(b["B4"]), stretch(b["B3"]), stretch(b["B2"])])
false_colour = np.dstack([stretch(b["B8"]), stretch(b["B4"]), stretch(b["B3"])])   # NIR-Red-Green: vegetation in red
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(true_colour); ax[0].set_title("True colour (B4-B3-B2)")
ax[1].imshow(false_colour); ax[1].set_title("False colour infrared (B8-B4-B3)")
for a in ax: a.axis("off")
plt.show()

### Spectral indices
Normalized differences reduce illumination effects and highlight one surface type. Each index ranges from −1 to 1.

| Index | Formula (Sentinel-2 bands) | High values mean |
|---|---|---|
| NDVI (Rouse et al., 1974) | (B8 − B4) / (B8 + B4) | Dense, healthy vegetation |
| NDWI (McFeeters, 1996) | (B3 − B8) / (B3 + B8) | Open water |
| NDBI (Zha et al., 2003) | (B11 − B8) / (B11 + B8) | Built-up and bare surfaces |

## Part B — Lab

**Task 1 — NDVI.** Compute the NDVI image `ndvi` (a 2-D NumPy array) from `b['B8']` and `b['B4']`.

In [ ]:
# YOUR CODE HERE
ndvi = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(ndvi.shape == (200, 200), "ndvi must be a 200 x 200 array")
check(ndvi[truth == 1].mean() > 0.6, "Fields should have NDVI > 0.6")
plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1); plt.colorbar(label="NDVI"); plt.title("NDVI"); plt.axis("off"); plt.show()

**Task 2 — NDWI and NDBI.** Compute `ndwi` and `ndbi` with the formulas in the table.

In [ ]:
# YOUR CODE HERE
ndwi = None
ndbi = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(ndwi[truth == 0].mean() > 0.3, "Water should have positive NDWI")
check(ndbi[truth == 3].mean() > ndbi[truth == 1].mean(), "Built-up should have higher NDBI than vegetation")

**Task 3 — Map water.** Create a boolean mask `water` where NDWI > 0, then compute the lake area in **hectares**
(`water_ha`; one 10 m pixel = 0.01 ha).

In [ ]:
# YOUR CODE HERE
water = None
water_ha = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
true_ha = (truth == 0).sum() * 0.01
print(f"Mapped {water_ha:.1f} ha vs true {true_ha:.1f} ha")
check(abs(water_ha - true_ha) / true_ha < 0.05, "Within 5% of the true lake area")

**Task 4 — A rule-based classifier.** Build `classes` (integers 0-3 like `truth`: 0 water, 1 vegetation, 2 bare soil,
3 built-up) from thresholds on your indices, then compute `accuracy`, the share of pixels equal to `truth`.

In [ ]:
# YOUR CODE HERE
classes = None
accuracy = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(accuracy > 0.85, "Tune your thresholds to reach at least 85%")

**❓ Question —** Rule-based thresholds worked here. Why would they fail across a whole province or between seasons? What does this suggest for Session 9?

*Your answer here.*

## Summary

- Passive optical sensors measure reflected sunlight; radar works through clouds.
- Choose the sensor by spatial, spectral, temporal and radiometric resolution.
- Normalized indices (NDVI, NDWI, NDBI) turn bands into interpretable variables — and good ML features.

**To go further:**

- ESA Sentinel-2 user guide: https://sentiwiki.copernicus.eu/web/s2-mission
- Cardille et al. (2024), *Cloud-Based Remote Sensing with Google Earth Engine*, chapters F1–F2 (open access).